In [ ]:
# Use only the package used here
import math
import random

In [ ]:
class Data:
  def __init__(self, sampleList = None, clsGeneList =None):
    self.sampleList = sampleList   # sample list
    self.clsGeneList = clsGeneList # candidate gene list for classification

class Cls:
  def __init__(self, clsResult = None, clsGene = None, cutoffValue = None):
    self.clsResult = clsResult     # classification (only inputted when the node is leaf node - 0: not determined, 1: normal, 2:disease)
    self.clsGene = clsGene         # gene of best classification (previously defined as int (wrong))
    self.cutoffValue = cutoffValue # cutoff value of the clsGene for classification

class Node:
  def __init__(self, data):
    self.data = data # microarray data (variable name was changed from "d")
    self.cls = Cls() # classification information
    self.high = None # son node (expression => cls_value)
    self.low = None  # son node (expression < cls_value)

In [ ]:
def replace(text):
  characters = "\n "
  for character in characters:
    return ''.join( x for x in text if x not in characters)

In [ ]:
def readDataFromFile(filename):
  sampleCls = {}
  exp = {}
  with open(filename) as file:
    sample_class = file.readline().split()
    for i in range(1, len(sample_class)):
      sampleCls[i] = sample_class[i] == 'non-tumor'
      exp[i] = {}
    for line in file:
      value = line.split()
      for i in range(1, len(sample_class)):
        exp[i][value[0]] = float(value[i])
  return sampleCls, exp

def readDEGFromFile(filename):
  deg = []
  with open(filename) as file:
    next(file)
    deg = [line.split()[0] for line in file]
  return deg

In [ ]:
def entropycal(fOrM,lAtT):
  if fOrM == 0 and lAtT == 0:
    return 0.00
  elif fOrM == 0:
    return 0.00
  elif lAtT == 0:
    return 0.00
  else:
    p = float(fOrM / (fOrM + lAtT))
    return -p*math.log(p, 2) - (1-p)*math.log(1 - p, 2)

In [ ]:
class DecisionTree:
  def __init__(self, sampleCls, exp):
    self.sampleCls = sampleCls # Sample property: normal(true) vs. cancer(false)
    self.exp = exp # microarray expression data
  
  def drawGraph(self, node, pre = ""):
    if node.cls.clsResult!=0: print("-[","Normal" if node.cls.clsResult ==1 else "Tumor","]")
    else:
      print("* ", node.cls.clsGene,"(", node.cls.cutoffValue, ")")
      print(pre, "|- >",end='')
      self.drawGraph(node.high, pre+"| ")
      print(pre, "|- <", end='')
      self.drawGraph(node.low, pre+"    ")

  def findCutoffValue(self, gene, sample_lists):
    # SuM_norm = 0 
    # SuM_cancer = 0 
    # cnt_n = 0
    # cnt_c = 0
    # for sample in sample_lists:
    #   if self.sampleCls[sample]:
    #     SuM_norm += self.exp[sample][gene]
    #     cnt_n += 1
    #   else:
    #     SuM_cancer += self.exp[sample][gene]
    #     cnt_c += 1

    #   if cnt_n == 0 or cnt_c == 0:
    #     cutoffvalue = None
    #   else:
    #     mean_norm = SuM_norm / cnt_n
    #     mean_cancer = SuM_cancer / cnt_c
    #     cutoffvalue = (mean_norm + mean_cancer) / 2
  
    normal_list = []
    cancer_list = []
    for sample in sample_lists:
        if self.sampleCls[sample]:
            normal_list.append(self.exp[sample][gene])
        else:
            cancer_list.append(self.exp[sample][gene])

    normal_list.sort()
    cancer_list.sort()
    n_len = len(normal_list)
    c_len = len(cancer_list)

    if n_len == 0 or c_len == 0:
      cutoffvalue = None
    else:
      median_n = float(normal_list[(n_len + 1) // 2 - 1])
      median_c = float(cancer_list[(c_len + 1)// 2 - 1])
      cutoffvalue = (median_n + median_c) / 2.0

    return cutoffvalue
  
  def getInfoGain(self, gene, cutoffValue, sample_lists):
    Norm, Cancer, Norm_h, Norm_l, Cancer_h, Cancer_l = 0,0,0,0,0,0
    for sample in sample_lists:
      if self.sampleCls[sample]:
        Norm+=1
        if self.exp[sample][gene] >= cutoffValue: Norm_h+=1
        else: Norm_l+=1
      elif not self.sampleCls[sample]:
        Cancer+=1
        if self.exp[sample][gene] >= cutoffValue: Cancer_h+=1
        else: Cancer_l+=1
  
    entro_1 = entropycal(Norm, Cancer)
    entro_h = entropycal(Norm_h, Cancer_h)
    entro_l = entropycal(Norm_l, Cancer_l)

    output = entro_1 - entro_h*(Norm_h + Cancer_h)/(Norm + Cancer) - entro_l*(Norm_l + Cancer_l)/(Norm + Cancer)
    return output

  def makeDecisionTree(self, node):
    countNum_Normal, countNum_Total, countNum_Disease = 0,0,0 # number of normal sample, number of total sample, number of disease sample
    for data in node.data.sampleList:
      countNum_Total+=1 # counting the toal number of the sample
      if self.sampleCls[data]: countNum_Normal+=1 # counting the total number of normal sample
      else: countNum_Disease+=1 # counting the total number of disease sample
  
    if countNum_Normal == countNum_Total: # if all the samples are normal
      node.cls.clsResult = 1 # change the classfication status to 'normal'
      return # and now terminate
    elif countNum_Disease == countNum_Total: # if all the samples are disease samples
      node.cls.clsResult = 2 # change the classification status to 'disease'
      return # and now finish the loop
  
    if len(node.data.clsGeneList) == 0: # if there is no more gene to be used
      if countNum_Normal > countNum_Total - countNum_Normal: node.cls.clsResult = 1 # and if the number of normal samples are more than the disease samples change the classification status to 'normal'
      else: node.cls.clsReuslt = 2 #if not, change the classification status to 'disease'
      return
    else:
      node.cls.clsResult = 0 # if the case above all does not fit, this node shall go more. Set the classification status to 'NULL'
      max_Gene = 0 # maximum value of information gain of the gene
      for clsGene in node.data.clsGeneList: # using the iterator, read all the genes in DEG list of the current node
        temp_COV = self.findCutoffValue(clsGene, node.data.sampleList) # [MainLabProblem] 1. find the cutoff value for the gene
        current_Gene = self.getInfoGain(clsGene, temp_COV, node.data.sampleList) # [MainLabProblem] 2. find the information gain for the gene
        if max_Gene < current_Gene:
          max_Gene = current_Gene
          Optimal_Gene = clsGene # [MainLabProblem] 3. find the gene with maximum information gain
      node.cls.clsGene = Optimal_Gene # [MainLabProblem] 4. the Optimal_Gene, which has the maximum information gain, will be the classification gene
      node.cls.cutoffValue = self.findCutoffValue(Optimal_Gene, node.data.sampleList) # [MainLabProblem]  5. find the cut off value for the gene
      lowstring = [] # this will save all the low expressed samples
      highstring = [] # this will save all the high expressed samples
      new_DEG = node.data.clsGeneList[:]
      new_DEG.remove(Optimal_Gene) # remove the gene that is used for classification
      for sample in node.data.sampleList:
        if self.exp[sample][Optimal_Gene] > node.cls.cutoffValue: # [MainLabProblem] 6. if the expression level of the sample is higher than the cut off value,
          highstring.append(sample) # put the high expressed samples in the highstring
        else: lowstring.append(sample) # if not,put the low expressed samples in the lowstirng
      
      node.high = Node(Data(highstring,new_DEG))
      node.low = Node(Data(lowstring, new_DEG))

      self.makeDecisionTree(node.low) # [MainLabProblem] 7. do the same in the low node. (recursively)
      self.makeDecisionTree(node.high) # [MainLabProblem] 8. do the same in the high node. (recursively)
  
  def predSample(self, sampleNo, Node):
    if Node.cls.clsResult != 0: # if the current node is a leaf node
      if Node.cls.clsResult == 1: # and if the current node says normal,
        torf = True # [MainLabProblem] 9. this sample is normal
      elif Node.cls.clsResult == 2: # else if the current node says disease,
        torf = False # [MainLabProblem] 10. this sample is disease

    elif Node.cls.clsResult == 0: # else if the current node is not a leaf node,
      if Node.cls.cutoffValue > self.exp[sampleNo][Node.cls.clsGene]: # and if the sample's expression level is lower than the cut off value
        torf = self.predSample(sampleNo, Node.low) # [MainLabProblem] 11. go to the low node
      else: torf = self.predSample(sampleNo, Node.high) # [MainLabProblem] 12. go to the high node 
    
    return torf #return whether the sample is disease or not
  
  def getAccuracy(self, sampleList, Node):
    # For the given sample list, test each samples and calculate accuracy.
    truepositive ,falsepositive ,truenegative ,falsenegative = 0,0,0,0

    for sample in sampleList:
      # [MainLabProblem] 13. Write your own code in here.
      # count truepositive, falsepositive, truenegative and falsenegative
      #_____________________________________
      # your code
      #_____________________________________
      if self.predSample(sample, Node) and self.sampleCls[sample]:
        truepositive += 1
      elif self.predSample(sample, Node) and not self.sampleCls[sample]:
        falsepositive += 1
      elif not self.predSample(sample, Node) and self.sampleCls[sample]:
        falsenegative += 1
      else:
        truenegative += 1
      accuracy = (truepositive + truenegative)/(truepositive + falsepositive + falsenegative + truenegative) # [MainLabProblem] 14. calculate accuracy
    return accuracy # return the accuracy

  def nFoldCV(self, n, d):
    Num_Total ,Accuracy_Avg = 0,0
    sample_map = {} # this map will determine us whether the samples are already used to make a subset or not
    for sample in d.sampleList: #for all samples
      sample_map[sample] = False #inset the sample number, and the value 'false' since we didn't use any of them yet.
      Num_Total+=1 #count the total number of samples
    
    countNum = 0 # count the number of samples that are used to make a sublist
    sublistNum = 0 # this will determine where to put the sample numbers
    sublist = {} # this will save all the sublist that is made in this function

    while countNum != Num_Total: # if the samples are not all used to make a sublist
      randNum = random.randint(1, Num_Total) # make any random number between 1~maximum number of the sample
      if sample_map[randNum] == False: # if the sample is not used to make a sublist,
        sublistNum = sublistNum % n + 1 # put it in a sublist (number between sublist #1 ~ #total number of sublist)
        if sublistNum in sublist.keys(): # if the sublist is not empty
          sublist[sublistNum].append(randNum) # push the sample number in the back of the sublist
          countNum+=1 # now the number of samples used in making a sublist increased
          sample_map[randNum] = True # change uesd sample
        else:
          sublist[sublistNum] = [randNum] # add a new list
          countNum+=1 # now the number of samples used in making a sublist increased
          sample_map[randNum] = True # change uesd sample
  
    for i in range(n):
      test_subset = sublist[i + 1] # [MainLabProblem] 15. one of the subset will be used as test set
      train_subset = [j for j in range(1, Num_Total+1)]
      for b in test_subset:
        train_subset.remove(b)

      node = Node(Data(train_subset, d.clsGeneList)) # [MainLabProblem] 16. set data in node
      self.makeDecisionTree(node) # [MainLabProblem] 17. using the DEG list and train subsets, construct a decision tree
      Accuracy_Avg += self.getAccuracy(test_subset, node) # [MainLabProblem] 18. determine the accuarcy of the decision tree, and add it to the variable "Accuracy_Avg"

    return Accuracy_Avg / n # divide the variable with number of sets that we've done

In [ ]:
#1. read data from file
sCls_train, exp_train = readDataFromFile('Lab10_GSE13164_train.txt')
sCls_test, exp_test = readDataFromFile('Lab10_GSE13164_test.txt')

dt = DecisionTree(sCls_train, exp_train) # [MainLabProblem] 19. read data using given input file (train) and transfer proper parameter. refer to '__init__'
dt_test = DecisionTree(sCls_test, exp_test) # [MainLabProblem] 20. read data using given input file (test) and transfer proper parameter. refer to '__init__'
deg = readDEGFromFile('Lab10_GSE13164_DEG.txt') # [MainLabProblem] 21. read data using given input file (DEG)

In [ ]:
#2. Prepare data part of root node for a parameter of function that generates decision tree
n = Node(Data(dt.sampleCls.keys(), deg))

In [ ]:
#3. Generate Decision Tree & Draw the tree from root to reaf
dt.makeDecisionTree(n)
dt.drawGraph(n)

In [ ]:
#4. Prepare whole data for cross validation
d = Data()
d.clsGeneList = n.data.clsGeneList
d.sampleList = n.data.sampleList
print("Accuracy for all the sample: ", dt.getAccuracy(n.data.sampleList, n))

In [ ]:
#5. Print 10-fold cross validation result 
print(dt.nFoldCV(10,d))

In [ ]:
#6. Print Accuracy for test set 
test_sampleList = []
print("\n\n Paste only Sample Number and Result Class. (Ex: 1:1 )")
for it3 in dt_test.sampleCls.keys():
  test_sampleList.append(it3)
  print(" Test sample class results : (Sample Number:Result Class Number)  ", len(test_sampleList), " :", dt_test.predSample(len(test_sampleList),n))

print("ACC for test set : ", dt_test.getAccuracy(test_sampleList, n))